# LAB: L'État et le Contexte d'un Agent
## Agent State and Context Management
### Master SDIA - Prof. RETAL SARA

Ce notebook couvre la gestion de l'état et du contexte dans les agents LangChain.

## 📚 Qu'est-ce que l'État et le Contexte?

### **Contexte (Context)**
- 📋 Informations structurées sur l'utilisateur/session
- 📊 Disponible pour les tools via `ToolRuntime`
- 🔒 En lecture seule par défaut
- 👤 Exemple: Préférences utilisateur, paramètres de session

### **État (State)**
- 💾 Données persistantes de l'agent
- 🔄 Peut être modifié pendant l'exécution
- 💭 Sauvegardé avec checkpointer
- 🎯 Exemple: Informations apprises, historique des actions

### Différences clés:

| Aspect | Contexte | État |
|--------|----------|------|
| Mutabilité | En lecture seule | Modifiable |
| Persistance | Pas de sauvegarde | Sauvegardé |
| Accès | Via ToolRuntime.context | Via AgentState |
| Scope | Requête unique | Entre requêtes |
| Cas d'usage | Config, préfs | Mémoire, historique |

### Avantages:
- ✅ Séparation des préoccupations
- ✅ Données structurées type-safe
- ✅ Persistance transparente
- ✅ Partage d'état entre tools
- ✅ Debugging plus facile

## ⚙️ SETUP - Installation et Configuration

In [ ]:
print("""
🚀 INSTALLATION DES DÉPENDANCES
="*70

Exécutez dans votre terminal:

  uv add langchain langchain-ollama langchain-core \\
          langgraph pydantic python-dotenv

Packages requis:
  ✓ langchain - Framework principal
  ✓ langchain-core - Core types
  ✓ langgraph - Graph et state management
  ✓ pydantic - Validation de données
  ✓ dataclasses - Classes structurées
""")

print("\nVérification des imports...\n")

from dataclasses import dataclass
from typing import Optional
import sys

print("✅ dataclasses: OK")

try:
    from pydantic import BaseModel
    print("✅ pydantic: OK")
except ImportError:
    print("❌ pydantic: À installer")

try:
    from langchain.agents import AgentState
    print("✅ langchain.agents: OK")
except ImportError:
    print("❌ langchain: À installer")

---

## 📋 PARTIE 1: Définir une classe de contexte structurée

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional

print("\n📋 CONTEXTE STRUCTURÉ")
print("="*70)

# Définir une classe de contexte
@dataclass
class ColorContext:
    """Contexte utilisateur pour les préférences de couleurs."""
    favourite_color: str = "blue"
    least_favourite_color: str = "yellow"
    user_id: str = "unknown"
    session_id: Optional[str] = None

print("✅ Classe ColorContext définie")
print(f"   Champs: {ColorContext.__dataclass_fields__.keys()}")
print()

# Exemple d'utilisation
context1 = ColorContext()
print(f"Context par défaut:")
print(f"  Favorite: {context1.favourite_color}")
print(f"  Least: {context1.least_favourite_color}")
print()

context2 = ColorContext(favourite_color="green", user_id="user123")
print(f"Context personnalisé:")
print(f"  Favorite: {context2.favourite_color}")
print(f"  User ID: {context2.user_id}")

### Autres types de contextes

In [ ]:
from dataclasses import dataclass
from typing import Dict, List
from datetime import datetime

print("\n📊 AUTRES CONTEXTES POSSIBLES\n")

# Contexte pour une application e-commerce
@dataclass
class ShoppingContext:
    """Contexte utilisateur pour un agent e-commerce."""
    user_id: str
    cart_items: List[str] = None
    total_budget: float = 1000.0
    preferred_categories: List[str] = None
    
print("✅ ShoppingContext:")
print("   Gère les informations d'achat")
print()

# Contexte pour une application médicale
@dataclass
class PatientContext:
    """Contexte patient pour un assistant médical."""
    patient_id: str
    age: int
    medical_history: List[str]
    current_medications: List[str]
    allergies: List[str]
    
print("✅ PatientContext:")
print("   Gère les informations médicales")
print()

# Contexte pour une application de support
@dataclass
class SupportContext:
    """Contexte pour un agent de support client."""
    ticket_id: str
    customer_name: str
    issue_type: str
    priority: str  # "low", "medium", "high"
    created_at: str
    customer_history: List[str] = None
    
print("✅ SupportContext:")
print("   Gère les tickets de support")

---

## 🤖 PARTIE 2: Agent sans contexte

In [ ]:
print("""
🤖 AGENT SANS CONTEXTE
="*70

L'agent sans contexte ne peut pas accéder aux informations utilisateur.

Code:

```python
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain.messages import HumanMessage

model = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

agent = create_agent(model=model)

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]}
)

print(response['messages'][-1].content)
```

Résultat:
  → L'agent ne sait pas quelle est votre couleur préférée
  → Il demandera à l'utilisateur de préciser
  → Pas d'accès aux données contextuelles

Inconvénients:
  ❌ Pas d'accès aux informations utilisateur
  ❌ Expérience personnalisée limitée
  ❌ L'agent doit demander les informations à chaque fois
""")

---

## 🎯 PARTIE 3: Agent avec contexte

In [ ]:
from langchain.tools import tool, ToolRuntime
from dataclasses import dataclass

print("""
🎯 AGENT AVEC CONTEXTE
="*70

L'agent avec contexte peut accéder aux informations via ToolRuntime.

Étape 1: Définir des tools qui accèdent au contexte
""")

@dataclass
class ColorContext:
    favourite_color: str = "blue"
    least_favourite_color: str = "yellow"

print("✅ ColorContext défini\n")

print("Étape 2: Créer des tools avec accès au contexte\n")

# Tool pour lire la couleur préférée
@tool
def get_favourite_color(runtime: ToolRuntime) -> str:
    """Get the favourite color of the user from context."""
    return runtime.context.favourite_color

# Tool pour lire la couleur moins préférée
@tool
def get_least_favourite_color(runtime: ToolRuntime) -> str:
    """Get the least favourite color of the user from context."""
    return runtime.context.least_favourite_color

print("✅ Tools créés:")
print(f"  - get_favourite_color")
print(f"  - get_least_favourite_color")
print()

print("Étape 3: Créer l'agent avec context_schema\n")

print("Code:")
print("""
```python
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[get_favourite_color, get_least_favourite_color],
    context_schema=ColorContext
)
```
""")

print("Utilisation:")
print("""
```python
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColorContext()  # ← Fournir le contexte
)
```
""")

### Avantages du contexte

In [ ]:
print("""
✨ AVANTAGES DU CONTEXTE
="*70

1. Informations structurées
   ✓ Type-safe (vérification à la compilation)
   ✓ Documentation claire
   ✓ Validation automatique

2. Réutilisabilité
   ✓ Les tools peuvent accéder au contexte
   ✓ Pas besoin de passer des paramètres
   ✓ Centré sur le contexte utilisateur

3. Personnalisation
   ✓ L'agent s'adapte au contexte
   ✓ Réponses pertinentes pour l'utilisateur
   ✓ Comportement configurable

4. Expérience utilisateur
   ✓ Pas de questions inutiles
   ✓ Réponses basées sur les préférences
   ✓ Service plus intelligent

Exemple de flux avec contexte:

    Utilisateur: "What is my favourite colour?"
         ↓
    Agent détecte la question
         ↓
    Tool get_favourite_color() appelé
         ↓
    ToolRuntime.context.favourite_color accédé
         ↓
    Retour: "Your favourite colour is blue"
""")

---

## 🔄 PARTIE 4: Changer le contexte

In [ ]:
print("""
🔄 CHANGEMENT DE CONTEXTE
="*70

Vous pouvez créer différents contextes pour le même agent.

Code:

```python
# Contexte 1: Contexte par défaut
response1 = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColorContext()  # blue (défaut)
)
print(response1['messages'][-1].content)
# → "Your favourite colour is blue"

# Contexte 2: Contexte personnalisé
response2 = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColorContext(favourite_color="green")  # green
)
print(response2['messages'][-1].content)
# → "Your favourite colour is green"

# Contexte 3: Autre utilisateur
response3 = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColorContext(favourite_color="red")  # red
)
print(response3['messages'][-1].content)
# → "Your favourite colour is red"
```

Cas d'usage:
  • Multi-utilisateur: Chaque utilisateur son contexte
  • Multi-session: Contexte par session
  • A/B Testing: Tester différentes configurations
  • Simulation: Tester avec différents scenarii
""")

print("\n✨ Avantage: Même agent, contextes différents!")

---

## 💾 PARTIE 5: État personnalisé (CustomState)

In [ ]:
from typing import List
from langchain.agents import AgentState

print("""
💾 ÉTAT PERSONNALISÉ (CUSTOM STATE)
="*70

L'état (State) est différent du contexte:

  Contexte:     En lecture seule, session courante
  État:         Modifiable, persistant entre requêtes

Créer un état personnalisé en héritage de AgentState:

```python
from langchain.agents import AgentState

class CustomState(AgentState):
    """État personnalisé pour l'agent."""
    favourite_color: str = "blue"  # ← Nouveau champ
    learned_facts: List[str] = []  # ← Liste de faits appris
    interaction_count: int = 0      # ← Compteur d'interactions
```

Champs hérités de AgentState:
  • messages: List[BaseMessage] - Historique des messages
  • metadata: dict - Métadonnées

Champs ajoutés:
  • favourite_color - Couleur préférée (state)
  • learned_facts - Faits appris pendant l'exécution
  • interaction_count - Nombre d'interactions
""")

print("\nDéfinition de CustomState:")
print()

class CustomState(AgentState):
    """État personnalisé pour l'agent couleur."""
    favourite_color: str = "blue"
    learned_facts: List[str] = None
    interaction_count: int = 0

print("✅ CustomState créé avec succès")
print(f"   Champs de base (AgentState): messages, metadata")
print(f"   Champs personnalisés: favourite_color, learned_facts, interaction_count")

---

## 🔧 PARTIE 6: Agent qui modifie l'état

In [ ]:
print("""
🔧 AGENT QUI MODIFIE L'ÉTAT
="*70

Les tools peuvent modifier l'état via Command.

Code exemple:

```python
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_color(new_color: str, runtime: ToolRuntime) -> Command:
    \"\"\"Update the favourite colour in the state.\"\"\"
    return Command(update={
        "favourite_color": new_color,
        "messages": [
            ToolMessage(
                f"Successfully updated favourite colour to {new_color}",
                tool_call_id=runtime.tool_call_id
            )
        ]
    })
```

Flux d'exécution:

    1. Utilisateur: "My favourite colour is green"
         ↓
    2. Agent détecte le changement
         ↓
    3. Tool update_favourite_color("green") appelé
         ↓
    4. Command retourné avec update={"favourite_color": "green"}
         ↓
    5. État mis à jour: favourite_color = "green"
         ↓
    6. État sauvegardé via checkpointer
         ↓
    7. Message de confirmation envoyé à l'utilisateur

Points clés:
  ✓ Command() pour modifier l'état
  ✓ update={} contient les modifications
  ✓ Checkpointer sauvegarde automatiquement
  ✓ ToolMessage pour la confirmation
""")

### Exemple complet avec state modification

In [ ]:
print("""
💡 EXEMPLE: Agent qui apprend

```python
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

# Créer l'agent avec state et checkpointer
agent = create_agent(
    model=model,
    tools=[update_favourite_color],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

# Première interaction: Apprendre la couleur
config = {"configurable": {"thread_id": "user123"}}

response1 = agent.invoke(
    {"messages": [HumanMessage(content="My favourite colour is green")]},
    config
)
print(response1['messages'][-1].content)
# → "Successfully updated favourite colour to green"

# Vérifier que l'état a été sauvegardé
print(f"État actuel: {response1}")
print(f"Couleur: {response1['favourite_color']}")
# → "Couleur: green"
```

Avantages:
  ✓ L'agent apprend de l'interaction
  ✓ L'état persiste entre requêtes
  ✓ L'agent se souvient pour les futures interactions
  ✓ Multi-utilisateur via thread_id
""")

---

## 📖 PARTIE 7: Agent qui récupère l'état

In [ ]:
print("""
📖 AGENT QUI RÉCUPÈRE L'ÉTAT
="*70

Les tools peuvent lire l'état sauvegardé.

Code:

```python
@tool
def read_favourite_color(runtime: ToolRuntime) -> str:
    \"\"\"Read the favourite colour from state.\"\"\"
    try:
        return runtime.state["favourite_color"]
    except KeyError:
        return "No favourite colour found in state"
```

Utilisation:

```python
# Créer l'agent
agent = create_agent(
    model=model,
    tools=[update_favourite_color, read_favourite_color],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

config = {"configurable": {"thread_id": "user123"}}

# Étape 1: Apprendre la couleur
response1 = agent.invoke(
    {"messages": [HumanMessage(content="My favourite colour is green")]},
    config
)
print(response1['messages'][-1].content)
# → "Successfully updated favourite colour to green"

# Étape 2: Récupérer la couleur sauvegardée
response2 = agent.invoke(
    {"messages": [HumanMessage(content="What's my favourite colour?")]},
    config  # Même thread_id = même état
)
print(response2['messages'][-1].content)
# → "Your favourite colour is green"
```

Flux complet:

    Session 1 (thread_id="user123"):
      • Message: "My favourite is green"
      • State: favourite_color = "green"
      • Sauvegardé via checkpointer
    
    Session 2 (même thread_id):
      • Message: "What's my favourite?"
      • Tool récupère: favourite_color = "green"
      • Réponse basée sur l'état sauvegardé

Cas d'usage:
  ✓ Conversation multi-tour
  ✓ Apprentissage de préférences
  ✓ Suivi de progrès
  ✓ Mémoire à long terme
""")

---

## 🔀 Résumé: Contexte vs État

In [ ]:
print("""
🔀 CONTEXTE vs ÉTAT
="*70

Tableau Comparatif:

┌──────────────────┬──────────────────┬──────────────────┐
│ Aspect           │ Contexte         │ État             │
├──────────────────┼──────────────────┼──────────────────┤
│ Mutabilité       │ Lecture seule    │ Modifiable       │
│ Persistance      │ Non persistant   │ Persistant       │
│ Scope            │ Requête          │ Entre requêtes   │
│ Accès            │ via ToolRuntime  │ via AgentState   │
│ Modification     │ Non             │ via Command()    │
│ Sauvegarde       │ Pas de save      │ Checkpointer    │
│ Cas d'usage      │ Config user      │ Mémoire agent    │
│ Exemple          │ Préférences      │ Faits appris     │
└──────────────────┴──────────────────┴──────────────────┘

Quand utiliser Contexte:
  ✓ Informations de l'utilisateur (ID, préférences)
  ✓ Configuration de session
  ✓ Paramètres qui ne changent pas
  ✓ Données sensibles (read-only)

Quand utiliser État:
  ✓ Informations apprises par l'agent
  ✓ Historique des interactions
  ✓ Compteurs et métriques
  ✓ Données qui doivent persister

Interaction:
    User Context
         ↓
    Agent (avec state)
         ↓
    Tools (lisent context, modifient state)
         ↓
    Checkpointer (sauvegarde state)
""")

---

## 💡 Cas d'Usage Pratiques

In [ ]:
print("""
💡 CAS D'USAGE PRATIQUES
="*70

1️⃣ Assistant Personnel
   Contexte: Préférences (langue, fuseau horaire)
   État: Tâches apprises, notes personnelles

2️⃣ Agent E-commerce
   Contexte: ID client, budget, catégories préférées
   État: Panier, historique d'achats, recommandations

3️⃣ Support Client
   Contexte: ID ticket, priorité, customer_history
   État: Étapes résolues, solutions proposées, feedback

4️⃣ Tuteur IA
   Contexte: Niveau d'étudiant, sujet
   État: Concepts maîtrisés, erreurs, score progress

5️⃣ Assistant de Santé
   Contexte: Informations patient, allergies, médicaments
   État: Symptômes rapportés, diagnostic, traitements suggérés

6️⃣ Agent de Recommandation
   Contexte: Profil utilisateur, budget
   État: Items vus, items aimés, raison du rejet

Avantage clé:
  → Expérience cohérente et intelligente
  → Agent qui apprend et s'améliore
  → Personnalisation basée sur l'utilisateur
  → Données persistantes et structurées
""")

---

## 📝 Résumé et Points Clés

In [ ]:
print("""
📝 RÉSUMÉ - LES POINTS CLÉS
="*70

✅ CONTEXTE (ColourContext):
   1. Classe avec @dataclass
   2. Défini dans agent via context_schema
   3. Accessible dans tools via ToolRuntime.context
   4. Lecture seule
   5. Changeable par requête

✅ ÉTAT (CustomState):
   1. Hérite de AgentState
   2. Défini dans agent via state_schema
   3. Accessible dans tools via ToolRuntime.state
   4. Modifiable via Command()
   5. Persistent via checkpointer

✅ OUTILS (Tools):
   1. Marqués avec @tool
   2. Reçoivent ToolRuntime comme paramètre
   3. Lisent contexte et état
   4. Modifient état via Command()
   5. Retournent ToolMessage ou Command

✅ PERSISTANCE:
   1. Checkpointer (InMemorySaver, etc)
   2. thread_id pour identifier la session
   3. L'état persiste entre appels
   4. Multi-utilisateur via thread_id différents

Workflow d'une conversation:

    Request 1 (thread_id="user1"):
      Context: user_id="user1", favorite="blue"
      State: (initial)
      Message: "My favorite is green"
      → State sauvegardé: favorite="green"
    
    Request 2 (thread_id="user1", même utilisateur):
      Context: user_id="user1", favorite="blue" (inchangé)
      State: favorite="green" (sauvegardé!)
      Message: "What's my favorite?"
      → Retour: "Your favorite is green"
""")

print("\n" + "="*70)
print("✨ Vous maîtrisez maintenant l'état et le contexte d'un agent!")
print("="*70)